# 02 — Transformação e Limpeza dos Dados

Pipeline: filtrar Influenza, tipar colunas, calcular faixa etária, agregar por UF/mês.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import polars as pl
import pyarrow.dataset as pds
import pyarrow.fs as fs
import pyarrow.compute as pc

from src.config import S3_ENDPOINT, S3_ACCESS_KEY, S3_SECRET_KEY, S3_REGION, S3_BUCKET, ANOS, COLUNAS_PNI
from src.transform import (
    filtra_influenza_pni, filtra_sih_sim_influenza,
    tipa_pni, faixa_etaria, adiciona_regiao, adiciona_mes_ano,
    agrega_doses, prepara_pni_completo,
)

In [ ]:
# Carregar e preparar PNI
s3 = fs.S3FileSystem(
    endpoint_override=S3_ENDPOINT,
    access_key=S3_ACCESS_KEY,
    secret_key=S3_SECRET_KEY,
    region=S3_REGION,
)

dataset = pds.dataset(S3_BUCKET, filesystem=s3, format="parquet", partitioning="hive")
table = dataset.to_table(
    filter=pds.field("ano").isin(ANOS),
    columns=COLUNAS_PNI,
)
print(f"Total registros PNI: {table.num_rows:,}")

In [ ]:
# Filtrar Influenza + período maio/2025 a maio/2026
from datetime import datetime

dt_inicio = pc.scalar("2025-05-01", type=pc.timestamp("s"))
dt_fim = pc.scalar("2026-05-31", type=pc.timestamp("s"))
dt_vacina = table.column("dt_vacina")
mask_data = pc.and_(pc.greater_equal(dt_vacina, dt_inicio), pc.less_equal(dt_vacina, dt_fim))

table_filtrada = table.filter(mask_data)
table_influenza = filtra_influenza_pni(table_filtrada)
print(f"Registros Influenza no período: {table_influenza.num_rows:,}")

In [ ]:
# Aplicar transformações
df = prepara_pni_completo(table)
print(f"Shape: {df.shape}")
print(f"Colunas: {list(df.columns)}")
df.head()

In [ ]:
# Agregar por UF + mês
doses_agg = agrega_doses(df, "uf_mes").to_pandas()
doses_agg.head(10)

In [ ]:
# Carregar SIH e filtrar Influenza
from pysus.online_data.SIH import download

df_sih = download("SP", 2025)
df_sih_influenza = filtra_sih_sim_influenza(df_sih, "DIAG_PRINC")
print(f"Internações Influenza SP/2025: {len(df_sih_influenza)}")
df_sih_influenza.head()

In [ ]:
# Salvar dados processados
df.to_parquet("../data/processed/pni_influenza.parquet", index=False)
doses_agg.to_csv("../data/processed/doses_agg_uf_mes.csv", index=False)
print("Dados salvos em data/processed/")

---
**Próximo notebook**: `03_analysis.ipynb` — análise completa